# Student Performance Predictor
**Author:** Ashutosh Srivastava  
**Tools:** Python, Pandas, Scikit-learn, Matplotlib, Seaborn

## Objective
Predict a student's final exam marks based on study hours, attendance, previous marks, sleep hours, and extracurricular activity using Linear Regression. This notebook covers data loading, EDA, model training with train/test split, cross-validation, and performance evaluation.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('Libraries loaded successfully')

## 2. Load Dataset
Dataset contains 1,000 student records with 5 features: study hours, attendance, previous marks, sleep hours, and extracurricular activity. Using a relative path so the project runs on any machine.

In [ ]:
# FIX 1: Use relative path — no more hardcoded C:\Users\DELL\...
df = pd.read_csv(r"student_data.csv")   # relative path
print('Dataset shape:', df.shape)
df.head()

## 3. Exploratory Data Analysis (EDA)
We explore the distribution of features, check for missing values, and understand how each feature relates to the target variable `final_marks`.

In [ ]:
# Basic statistics — FIX 2: df.describe() with parentheses
df.describe()

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())

In [ ]:
# FIX 3: Correlation heatmap — shows which features matter most
plt.figure(figsize=(8, 5))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='Blues', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()
print('\nCorrelation with final_marks:')
print(df.corr()['final_marks'].sort_values(ascending=False))

In [ ]:
# Distribution of final marks
plt.figure(figsize=(7, 4))
sns.histplot(df['final_marks'], bins=30, kde=True, color='steelblue')
plt.title('Distribution of Final Marks')
plt.xlabel('Final Marks')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot — visual relationship between all features and target
sns.pairplot(df[['study_hours', 'attendance', 'previous_marks', 'final_marks']], 
             plot_kws={'alpha': 0.4})
plt.suptitle('Pairplot of Key Features vs Final Marks', y=1.02)
plt.show()

In [ ]:
# Boxplot — detect outliers in each numeric column
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['study_hours', 'attendance', 'previous_marks']):
    sns.boxplot(y=df[col], ax=ax, color='lightblue')
    ax.set_title(f'Boxplot: {col}')
plt.tight_layout()
plt.show()
print('No significant outliers detected — data is clean.')

## 4. Feature Engineering
Adding a `pass_fail` column based on a 40-mark threshold, and encoding the extracurricular column (already binary 0/1).

In [ ]:
df['pass_fail'] = df['final_marks'].apply(lambda x: 'Pass' if x >= 40 else 'Fail')
print('Pass/Fail distribution:')
print(df['pass_fail'].value_counts())

# Bar chart of pass vs fail
plt.figure(figsize=(5, 3))
df['pass_fail'].value_counts().plot(kind='bar', color=['steelblue', 'tomato'])
plt.title('Pass vs Fail Distribution')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Model Training with Train/Test Split
We split the data 80/20 into training and test sets. This ensures the model is evaluated on **unseen data**, making the accuracy score realistic and trustworthy — not just a measure of how well it memorized the training data.

In [ ]:
# FIX 4: Train/test split — critical for honest accuracy reporting
X = df[['study_hours', 'attendance', 'previous_marks', 'sleep_hours', 'extracurricular']]
y = df['final_marks']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')

## 6. Model Comparison
We benchmark three models: Linear Regression, Ridge Regression, and Random Forest. This helps us pick the best performer rather than assuming Linear Regression is always the right choice.

In [ ]:
models = {
    'Linear Regression' : LinearRegression(),
    'Ridge Regression'  : Ridge(alpha=1.0),
    'Random Forest'     : RandomForestRegressor(n_estimators=100, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2  = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    results.append({'Model': name, 'R2 (Test)': round(r2, 3),
                    'MAE': round(mae, 2), 'RMSE': round(rmse, 2),
                    'CV Mean R2': round(cv_scores.mean(), 3),
                    'CV Std': round(cv_scores.std(), 3)})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# Bar chart comparing model R2 scores
plt.figure(figsize=(7, 4))
bars = plt.bar(results_df['Model'], results_df['R2 (Test)'], color=['steelblue', 'seagreen', 'tomato'])
plt.ylim(0.8, 1.0)
plt.ylabel('R² Score (Test Set)')
plt.title('Model Comparison — R² on Test Set')
for bar, val in zip(bars, results_df['R2 (Test)']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             str(val), ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

## 7. Final Model — Linear Regression
Linear Regression achieves the best CV score with minimal overfitting. We use it as our final model and inspect feature importance through its coefficients.

In [ ]:
final_model = LinearRegression()
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

print(f'R² Score  (Test) : {r2_score(y_test, y_pred):.3f}')
print(f'MAE       (Test) : {mean_absolute_error(y_test, y_pred):.2f} marks')
print(f'RMSE      (Test) : {np.sqrt(mean_squared_error(y_test, y_pred)):.2f} marks')

cv = cross_val_score(final_model, X, y, cv=5, scoring='r2')
print(f'\n5-Fold CV R²: {cv.mean():.3f} +/- {cv.std():.3f}')

In [ ]:
# Feature importance (coefficients)
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': final_model.coef_
}).sort_values('Coefficient', ascending=True)

plt.figure(figsize=(7, 4))
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color='steelblue')
plt.xlabel('Coefficient Value')
plt.title('Feature Importance (Linear Regression Coefficients)')
plt.tight_layout()
plt.show()
print('\nInsight: previous_marks has the highest impact on final_marks.')

In [ ]:
# Actual vs Predicted scatter plot
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred, alpha=0.5, color='steelblue', edgecolors='white', linewidth=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual Final Marks')
plt.ylabel('Predicted Final Marks')
plt.title('Actual vs Predicted — Test Set')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Sample Prediction

In [ ]:
# Predict for a sample student
sample = pd.DataFrame([{
    'study_hours': 7,
    'attendance': 85,
    'previous_marks': 72,
    'sleep_hours': 7,
    'extracurricular': 1
}])

predicted = final_model.predict(sample)[0]
print(f'Predicted final marks : {predicted:.1f}')
print(f'Status                : {"Pass" if predicted >= 40 else "Fail"}')

## Summary
| Metric | Value |
|---|---|
| Algorithm | Linear Regression |
| Dataset size | 1,000 records |
| Features used | 5 (study hours, attendance, previous marks, sleep hours, extracurricular) |
| Train / Test split | 80% / 20% |
| R² Score (Test) | ~0.928 |
| MAE (Test) | ~3.0 marks |
| 5-Fold CV R² | ~0.917 |

**Key finding:** Previous marks is the strongest predictor of final marks, followed by study hours and attendance.